In [1]:
import numpy as np
import re

In [12]:
data="""
우리말이 중국과 달라 글과 글자와 서로 통하지 않으니, 이런 까닭으로 어리석은 백성이 말하고자 하는 바가 있어도 결국 자신의 뜻을 막힘없이 펴지 못하는 사람이 많다. 내가 이를 위하여 가엾이 여겨 새로 스물여덟 자를 만드니 모든 사람들로 하여금 쉽게 익혀 날마다 쓰는 데 편하게 하고자 한다.
"""

In [13]:
def data_preprocessing(data):
    data = re.sub('[^가-힣]', ' ',data)
    tokens = data.split()
    vocab = list(set(tokens))
    vocab_size = len(vocab)

    word_to_ix= {word: i for i, word in enumerate(vocab)}
    ix_to_word= {i: word for i, word in enumerate(vocab)}
    
    return tokens, vocab_size, word_to_ix, ix_to_word


In [14]:
# 가중치 초기화 함수
def init_weights(h_size, vocab_size):
    # U: 입력층에서 은닉층으로 가는 가중치
    U = np.random.randn(h_size, vocab_size) * 0.01
    # W: 은닉층에서 다음 시점의 은닉층으로 가는 가중치 (Recurrent)
    W = np.random.randn(h_size, h_size) * 0.01
    # V: 은닉층에서 출력층으로 가는 가중치
    V = np.random.randn(vocab_size, h_size) * 0.01
    return U, W, V

In [15]:
# 순전파 함수
def feedforward(inputs, targets, hprev):
    loss = 0
    xs, hs, ps, ys = {}, {}, {}, {}
    hs[-1] = np.copy(hprev)
    
    for i in range(seq_len):
        # 입력 데이터를 One-hot vector로 변환
        xs[i] = np.zeros((vocab_size, 1))
        xs[i][inputs[i]] = 1 # 각각의 word에 대한 one hot coding
        
        # 은닉 상태(Hidden State) 계산: tanh(U*x + W*h_prev)
        hs[i] = np.tanh(np.dot(U, xs[i]) + np.dot(W, hs[i - 1]))
        
        # 출력 계산
        ys[i] = np.dot(V, hs[i])
        
        # Softmax 계산을 통해 확률분포 산출
        ps[i] = np.exp(ys[i]) / np.sum(np.exp(ys[i]))
        
        # Cross-entropy 손실 계산
        loss += -np.log(ps[i][targets[i], 0])
        
    return loss, ps, hs, xs

In [16]:
def backward(ps, hs, xs):
    # Backward propagation through time (BPTT)
    # 처음에 모든 가중치들은 0으로 설정
    dV = np.zeros(V.shape)
    dW = np.zeros(W.shape)
    dU = np.zeros(U.shape)

    for i in range(seq_len)[::-1]:
        output = np.zeros((vocab_size, 1))
        output[targets[i]] = 1
        
        # Softmax 출력과 실제 정답의 차이 (y_hat - y)
        ps[i] = ps[i] - output.reshape(-1, 1)
        
        # 매번 i 스텝에서 dL/dVi를 구하기
        # (y_hat - y) @ hs.T - for each step
        dV_step_i = ps[i] @ (hs[i]).T
        dV = dV + dV_step_i # dL/dVi를 다 더하기

        # 각 i별로 v와 w를 구하기 위해서는 
        # 먼저 공통적으로 계산되는 부분을 delta로 해서 계산해두고
        # i번째 스텝에서 공통적으로 사용될 delta
        # delta = (V.T @ dy) * (1 - h^2)
        delta_recent = (V.T @ ps[i]) * (1 - hs[i] ** 2)

        # 시간을 거슬러 올라가서 dL/dW와 dL/dU를 구함
        for j in range(i + 1)[::-1]:
            # dW_ij = delta @ hs[j-1].T
            dw_ij = delta_recent @ hs[j - 1].T
            dW = dW + dw_ij
            
            # dU_ij = delta @ xs[j].T
            dU_ij = delta_recent @ xs[j].reshape(1, -1)
            dU = dU + dU_ij
            
            # 그리고 다음번 j번째 타임에서 공통적으로 계산할 delta를 업데이트
            # delta = (W.T @ delta) * (1 - h_prev^2)
            delta_recent = (W.T @ delta_recent) * (1 - hs[j - 1] ** 2)

    # Gradient Clipping: 기울기 폭주(Exploding Gradients)를 방지하기 위해 값을 -1 ~ 1 사이로 제한
    for d in [dU, dW, dV]:
        np.clip(d, -1, 1, out=d)

    return dU, dW, dV, hs[len(inputs) - 1]

In [17]:
def predict(word, length):
    # 초기 입력 단어를 One-hot vector로 변환
    x = np.zeros((vocab_size, 1))
    x[word_to_ix[word]] = 1
    ixes = []
    
    # 은닉 상태(Hidden State)를 0으로 초기화
    h = np.zeros((h_size, 1))

    for t in range(length):
        # 1. 현재 입력(x)과 이전 상태(h)로 새로운 은닉 상태 계산
        h = np.tanh(np.dot(U, x) + np.dot(W, h))
        
        # 2. 출력층 계산
        y = np.dot(V, h)
        
        # 3. 소프트맥스를 통해 확률분포 산출
        p = np.exp(y) / np.sum(np.exp(y))
        
        # 4. 가장 높은 확률을 가진 단어의 index를 선택 (argmax)
        ix = np.argmax(p)
        
        # 5. 다음번 input x를 준비 (방금 예측한 단어를 다음 입력으로 사용)
        x = np.zeros((vocab_size, 1))
        x[ix] = 1
        
        # 결과 인덱스 저장
        ixes.append(ix)

    # 인덱스 리스트를 실제 단어로 변환하여 이어붙임
    pred_words = ' '.join(ix_to_word[i] for i in ixes)
    return pred_words

In [18]:
# 기본적인 parameters
epochs = 10000
h_size = 100       # Hidden state의 크기 (은닉층 노드 수)
seq_len = 3        # 한 번에 학습할 시퀀스의 길이
learning_rate = 1e-2

In [19]:
# 데이터 전처리 함수 호출
# 데이터셋(data)을 넣으면 토큰 리스트, 단어 사전 크기, 인덱스 변환 사전 등을 반환합니다.
tokens, vocab_size, word_to_ix, ix_to_word = data_preprocessing(data)

In [22]:
U, W, V = init_weights(h_size, vocab_size)

In [23]:
p=0
hprev = np.zeros((h_size,1))

for epoch in range(epochs):

    for p in range(len(tokens)-seq_len):
        inputs = [word_to_ix[tok] for tok in tokens[p:p+seq_len]]
        targets = [word_to_ix[tok] for tok in tokens [p+1:p+seq_len+1]]

        loss, ps, hs, xs = feedforward(inputs, targets, hprev)
        dU, dW, dV, hprev = backward(ps, hs, xs)

        W -= learning_rate*dW
        U -= learning_rate*dU
        V -= learning_rate*dV

    if epoch % 100 == 0:
        print(f'epoch {epoch}, loss: {loss}')

epoch 0, loss: 11.351801068738311
epoch 100, loss: 3.301705548535816
epoch 200, loss: 0.35537025500656655
epoch 300, loss: 0.13981097501444467
epoch 400, loss: 0.08546601990738618
epoch 500, loss: 0.06206069405251741
epoch 600, loss: 0.04902603519147046
epoch 700, loss: 0.04063328075973739
epoch 800, loss: 0.0345633635052681
epoch 900, loss: 0.029875698683936865
epoch 1000, loss: 0.026157130298691017
epoch 1100, loss: 0.023163500879113813
epoch 1200, loss: 0.02072335313813073
epoch 1300, loss: 0.018710385516398226
epoch 1400, loss: 0.01703023718682615
epoch 1500, loss: 0.015611737995556832
epoch 1600, loss: 0.014400677418051422
epoch 1700, loss: 0.013355491751655155
epoch 1800, loss: 0.012444246120472482
epoch 1900, loss: 0.011642375137964837
epoch 2000, loss: 0.010930916579117326
epoch 2100, loss: 0.010295132479899272
epoch 2200, loss: 0.00972345169329775
epoch 2300, loss: 0.00920667310001981
epoch 2400, loss: 0.008737374090996095
epoch 2500, loss: 0.008309476807239594
epoch 2600, los

In [24]:
while 1:
    try:
        # 사용자로부터 단어 또는 문장을 입력받습니다.
        user_input = input("input: ")
        
        # 'break'라고 입력하면 무한 루프를 종료합니다.
        if user_input == 'break':
            break
        
        # 앞서 정의한 predict 함수를 호출하여 40자 길이의 텍스트를 생성합니다.
        response = predict(user_input, 40)
        print(response)
        
    except:
        # 입력 단어가 사전에 없거나 예측 중 오류가 발생할 경우 실행됩니다.
        print('Uh oh try again!')

중국과 달라 글과 달라 글과 글자와 글과 글자와 서로 글자와 서로 통하지 않으니 통하지 않으니 이런 않으니 이런 까닭으로 이런 까닭으로 어리석은 까닭으로 어리석은 백성이 어리석은 백성이 말하고자 백성이 말하고자 하는 바가 하는 바가 있어도 바가 있어도 결국 있어도 결국
